# Direct String Matching and Result Contracts


```{contents}
:local:
:depth: 2
```


```{index} Define What Counts as a Match Before Choosing an Algorithm
```



```{index} Define What Counts as a Match Before Choosing an Algorithm
```

## Define What Counts as a Match Before Choosing an Algorithm



A message filter searches text for a literal pattern. Decide whether matching is case-sensitive, whether all overlapping starts are required, and how an empty pattern is treated. These choices determine the correct result before any comparison-count claim can be made. The core examples compare ordinal UTF-16 code units; a returned position is an offset in that representation, not necessarily a user-perceived character count.

For text ABABA and pattern ABA, the starts are zero and two. Moving the candidate start by the pattern length after the first match would miss the overlapping occurrence at two. A direct matcher instead considers every candidate start whose remaining text is long enough, comparing pattern elements until a mismatch or a complete match.

| Candidate start | Comparison outcome | Report it? |
| --- | --- | --- |
| 0 | ABA matches ABA | Yes |
| 1 | B differs from A immediately | No |
| 2 | ABA matches ABA | Yes |

The direct algorithm's simplicity makes it a useful independent reference for more elaborate matching. Its contract must still handle an empty pattern. Under the all-boundaries convention used here, an empty pattern matches before the first code unit, between adjacent code units, and after the final code unit. Thus text of length n has n+1 empty-pattern starts. Another API can choose another contract, but comparisons must use the same one.

For related explanation, see the further reading below.[^development-reading]


```{index} Use Mismatches as Evidence
```



```{index} Use Mismatches as Evidence
```

## Use Mismatches as Evidence



A mismatch at one candidate start establishes only that this candidate fails. It does not prove that the text contains no later occurrence. The outer loop advances the start; the inner loop advances the pattern position. Confusing those two positions can skip starts or compare the wrong slices.

With text length n and pattern length m, a direct comparison model can perform up to proportional to n*m work for nonempty patterns within the text. A long shared prefix followed by a mismatch can repeat work at many starts. The next lesson develops how prefix information can avoid some repetition. For short patterns or a small number of queries, a clear direct approach or a suitable library operation can still be a reasonable engineering choice; a more sophisticated algorithm is not required merely because it exists.

Guided check: find all starts of AA in AAAA, then an absent pattern longer than the text, then the empty pattern. Derive expected outputs manually. Use those cases when checking the preserved implementations so that a fast candidate cannot quietly change overlap or boundary semantics. Keep weighted-graph hop/cost reminders as links to Chapters 20 and 24; they are separate result-contract examples, not prerequisites for reading a string matcher.


```{index} Shortest Paths and String Matching
```

## Shortest Paths and String Matching



```{contents}
:local:
:depth: 2
```

Algorithm assumptions determine what an answer means. BFS minimizes hops, Dijkstra minimizes nonnegative total weights, and a string matcher needs a precise equality and occurrence policy. Reusing justified information can eliminate repeated work without changing the contract.

```{index} Minimum Hops Is Not Minimum Cost
```

## Minimum Hops Is Not Minimum Cost




```{index} Minimum Hops Is Not Minimum Cost
```

Suppose HQ→Client has one edge costing 100, while HQ→Helpdesk→Client costs 1+1. BFS chooses one hop; a nonnegative weighted shortest-path algorithm chooses cost two. Chapter 24 provides the complete guarded Dijkstra implementation. Negative weights require another algorithm, and a reachable negative cycle on a route to the target can mean no finite minimum exists.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var graph=new GraphWalk(new Dictionary<string,string[]>{["HQ"]=new[]{"Client","Helpdesk"},["Helpdesk"]=new[]{"Client"},["Client"]=Array.Empty<string>()});
var bfs=graph.Bfs("HQ");
Console.WriteLine($"minimum-hop route={string.Join("->",bfs.PathTo("Client"))}, hops={bfs.Distance["Client"]}");
Console.WriteLine("that route costs 100; feasible two-hop route costs 2");
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


This is a verified objective counterexample, not a reason to call BFS incorrect. Declare units, direction, permitted weights, and whether the requested result is reachability, edge count, total cost, or all destinations before selecting an algorithm.

```{index} Direct Matching with an Explicit Contract
```

## Direct Matching with an Explicit Contract




```{index} Direct Matching with an Explicit Contract
```

Search exact ordinal UTF-16 code units, returning every start position, including overlapping matches. A missing nonempty pattern returns an empty array. An empty pattern matches each boundary 0..text.Length; null arguments are rejected. Derived first-match clients return the first position or −1. Counts include every executed code-unit equality test, including mismatches.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

foreach(var pair in new[]{("aaaaa","aaa"),("abracadabra","cad"),("abc",""),("","x")})
{
 var result=PatternScan.Direct(pair.Item1,pair.Item2);
 Console.WriteLine($"text=[{pair.Item1}], pattern=[{pair.Item2}], positions=[{string.Join(",",result.Positions)}], comparisons={result.Comparisons}");
}
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


The loop tests each legal start once. At offset j, the first j code units match; a mismatch rules out that start, while j=m proves an occurrence. For n text units and m>0 pattern units, worst-case comparison work is O((n−m+1)m) when m≤n, commonly written O(nm); m>n gives zero tests. Returned r positions cost O(r) storage. Empty patterns require O(n) output despite zero equality tests.

```{rubric} Footnotes
```


[^development-reading]: Further reading: [Sedgewick and Wayne, Strings](https://algs4.cs.princeton.edu/50strings/). The worked business scenarios and guided reasoning here are original teaching examples.
